# 📊 PRÁCTICA: Manipulación de DataFrames en PySpark  
**Fuente:** NYC Taxi – Yellow Tripdata (2025-01, 2025-02, 2025-03)  
**Tecnología:** PySpark DataFrames  

---

## 1️⃣ Carga de datos

Se creó una sesión de Spark en modo local y se cargaron tres archivos Parquet correspondientes a los meses enero, febrero y marzo de 2025.

Los datos fueron almacenados en un **DataFrame llamado `df`**.

Se verificó:

- Que el DataFrame cargara correctamente.
- Las columnas disponibles.
- El esquema de los datos (`printSchema()`).
- El total de registros.

---

## 2️⃣ Uso de DataFrames para manipulación

Se trabajó con las siguientes columnas clave:

- `trip_distance`
- `fare_amount`
- `total_amount`
- `payment_type`

Se creó una vista base seleccionando solo estas columnas y se realizó:

- Selección de columnas específicas.
- Renombrado de `trip_distance` a `trip_distance_miles`.

Esto permite trabajar con un subconjunto más controlado de variables.

---

## 3️⃣ Modificación de datos

Se realizaron transformaciones para asegurar consistencia:

- Conversión explícita de tipos (`cast`) a:
  - `double` para montos y distancias.
  - `int` para `payment_type`.
- Redondeo de:
  - `fare_amount`
  - `total_amount`

Estas modificaciones mejoran la estabilidad y presentación de los datos.

---

## 4️⃣ Agregar nuevas columnas calculadas

Se crearon nuevas variables derivadas:

### 🔹 `tip_amount`
Calculada como:

```
total_amount - fare_amount
```

Representa una estimación de propina o cargos adicionales.

---

### 🔹 `fare_per_mile`
Calculada como:

```
fare_amount / trip_distance_miles
```

Se incluyó validación para evitar división entre cero.

---

### 🔹 `payment_label`
Se transformó `payment_type` en una etiqueta categórica:

- 1 → "Card"
- 2 → "Cash"
- Otro → "Other"

Esto facilita análisis posteriores por tipo de pago.

---

## 5️⃣ Filtrado de datos

Se filtraron registros válidos usando las siguientes condiciones:

- `trip_distance_miles > 0`
- `fare_amount > 0`
- `total_amount > 0`

Esto elimina registros inconsistentes o erróneos.

También se realizaron filtros adicionales:

- Top 10 viajes con mayor `total_amount`.
- Viajes pagados con tarjeta.
- Viajes cortos (≤ 1 milla).
- Viajes largos (≥ 10 millas).

---

## 6️⃣ Análisis descriptivo con DataFrames

Se calcularon estadísticas básicas:

- Conteo total
- Promedio de distancia
- Promedio de tarifa
- Promedio de total
- Promedio de costo por milla

También se realizó agrupación por tipo de pago (`groupBy`) para obtener:

- Número de viajes por método de pago
- Promedio de distancia
- Promedio de monto total

---

# ✅ Conclusión

En esta práctica se utilizaron **DataFrames de PySpark** para:

✔ Manipular filas y columnas  
✔ Modificar datos  
✔ Crear nuevas variables calculadas  
✔ Filtrar resultados  
✔ Realizar análisis descriptivo  

Se demuestra el uso completo del API de DataFrames para procesamiento distribuido de datos masivos.

In [1]:
# ============================================================
# PRÁCTICA: RDD EN PYSPARK (ESTADÍSTICAS DESCRIPTIVAS BÁSICAS)
# Fuente: NYC Taxi (Parquet) - Yellow Tripdata (2025-01..03)
# ============================================================

import os, sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ===============================
# 0) SPARK SESSION (tu configuración)
# ===============================
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PRACTICA_DF_PYSPARK")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
    .config("spark.executor.extraJavaOptions", "-Djava.net.preferIPv4Stack=true")
    # estabilidad
    .config("spark.python.worker.faulthandler.enabled", "true")
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
    .config("spark.python.worker.reuse", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

print("Spark listo ✅")
print("Python:", sys.executable)

# ===============================
# 1) CARGA DE LOS ARCHIVOS PARQUET
# ===============================
ruta_base = r"C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\6 - DATOS MASIVOS"

df = spark.read.parquet(
    f"{ruta_base}/yellow_tripdata_2025-01.parquet",
    f"{ruta_base}/yellow_tripdata_2025-02.parquet",
    f"{ruta_base}/yellow_tripdata_2025-03.parquet"
)

print("DataFrame cargado correctamente ✅")
print("Columnas disponibles:", df.columns)

# ============================================================
# 2) USAR DATAFRAMES PARA MANIPULAR DATOS (INSPECCIÓN BÁSICA)
# ============================================================

# Esquema y muestra
df.printSchema()
df.select(df.columns[:10]).show(5, truncate=False)  # muestra rápida (primeras 10 columnas)

# Conteo total (formato bonito con comas)
total_registros = df.count()
print(f"Total de registros: {total_registros:,}")

# ============================================================
# 3) MANIPULACIÓN DE FILAS Y COLUMNAS (SELECCIÓN / RENOMBRE)
#    * Usamos lo que ya venías trabajando: trip_distance, fare_amount, total_amount, payment_type
# ============================================================

cols_base = ["trip_distance", "fare_amount", "total_amount", "payment_type"]

df_base = (
    df.select(*cols_base)
      .withColumnRenamed("trip_distance", "trip_distance_miles")  # ejemplo de renombre
)

print("Vista base creada ✅")
df_base.show(5, truncate=False)

# ============================================================
# 4) MODIFICAR DATOS (CAST, LIMPIEZA, REDONDEO)
# ============================================================

df_mod = (
    df_base
    # Asegurar tipos numéricos (evita problemas por inferencia rara)
    .withColumn("trip_distance_miles", F.col("trip_distance_miles").cast("double"))
    .withColumn("fare_amount", F.col("fare_amount").cast("double"))
    .withColumn("total_amount", F.col("total_amount").cast("double"))
    .withColumn("payment_type", F.col("payment_type").cast("int"))
    # Redondeo (por presentación)
    .withColumn("fare_amount", F.round(F.col("fare_amount"), 2))
    .withColumn("total_amount", F.round(F.col("total_amount"), 2))
)

print("Datos modificados (cast + redondeo) ✅")
df_mod.show(5, truncate=False)

# ============================================================
# 5) AGREGAR NUEVAS COLUMNAS CALCULADAS
# ============================================================

df_calc = (
    df_mod
    # Tip estimada (si total incluye extras/propina)
    .withColumn("tip_amount", F.round(F.col("total_amount") - F.col("fare_amount"), 2))
    # Costo por milla (evitar división por 0)
    .withColumn(
        "fare_per_mile",
        F.when(F.col("trip_distance_miles") > 0, F.col("fare_amount") / F.col("trip_distance_miles"))
         .otherwise(F.lit(None))
    )
    .withColumn("fare_per_mile", F.round(F.col("fare_per_mile"), 2))
    # Etiqueta simple de pago (ejemplo de categorización)
    .withColumn(
        "payment_label",
        F.when(F.col("payment_type") == 1, F.lit("Card"))
         .when(F.col("payment_type") == 2, F.lit("Cash"))
         .otherwise(F.lit("Other"))
    )
)

print("Columnas calculadas agregadas ✅")
df_calc.show(5, truncate=False)

# ============================================================
# 6) FILTRAR RESULTADOS (REGISTROS VÁLIDOS + EJEMPLOS)
#    * Esto empata con lo que ya hiciste: > 0 en distance/fare/total
# ============================================================

df_filtrado = (
    df_calc
    .filter(F.col("trip_distance_miles") > 0)
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("total_amount") > 0)
)

validos = df_filtrado.count()
print(f"Registros válidos: {validos:,}")

# Ejemplo 1: top 10 viajes por total_amount
df_top_total = df_filtrado.orderBy(F.col("total_amount").desc())
df_top_total.show(10, truncate=False)

# Ejemplo 2: solo pagos con tarjeta
df_card = df_filtrado.filter(F.col("payment_type") == 1)
print(f"Viajes pagados con tarjeta: {df_card.count():,}")

# Ejemplo 3: viajes "cortos" (<= 1 milla) y "largos" (>= 10 millas)
df_short = df_filtrado.filter(F.col("trip_distance_miles") <= 1)
df_long  = df_filtrado.filter(F.col("trip_distance_miles") >= 10)
print(f"Viajes cortos (<=1 mi): {df_short.count():,}")
print(f"Viajes largos (>=10 mi): {df_long.count():,}")

# ============================================================
# 7) RESUMEN RÁPIDO (DESCRIPTIVOS CON DATAFRAME)
# ============================================================

# Estadísticas básicas
(df_filtrado
 .select(
     F.count("*").alias("n"),
     F.avg("trip_distance_miles").alias("avg_trip_distance"),
     F.avg("fare_amount").alias("avg_fare_amount"),
     F.avg("total_amount").alias("avg_total_amount"),
     F.avg("fare_per_mile").alias("avg_fare_per_mile")
 )
 .show(truncate=False)
)

# Por tipo de pago (agrupación)
(df_filtrado
 .groupBy("payment_label")
 .agg(
     F.count("*").alias("n"),
     F.avg("trip_distance_miles").alias("avg_distance"),
     F.avg("total_amount").alias("avg_total")
 )
 .orderBy(F.col("n").desc())
 .show(truncate=False)
)

print("Práctica de DataFrames completada ✅")


# ===============================
# 8) CERRAR SPARK
# ===============================
spark.stop()
print("\nSpark detenido ✅")

Spark listo ✅
Python: C:\Users\Oscar Ferreira\venv_pyspark\Scripts\python.exe
DataFrame cargado correctamente ✅
Columnas disponibles: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = tru